# CHITRA on Colab: Render-and-Match with RoMa v2

**CHITRA** (Chandrayaan High-precision Image Transformation & Registration Algorithm), SIH 2026 · PS 26166 · Team Death Eaters.

This notebook reproduces the measured results on a free GPU:
1. installs CHITRA with the RoMa v2 (DINOv3) dense matcher,
2. downloads the public LROC Apollo 11 DTM + orthophotos (~230 MB),
3. runs the exact-truth benchmark (SIFT vs RoMa v2 vs full CHITRA, Sun-azimuth change 30–150°, 1× and 4× scale),
4. optionally registers your own Chandrayaan-2 crop (PRADAN data can't be redistributed, so upload it).

**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*. It also runs on CPU, more slowly.

In [ ]:
!nvidia-smi -L || echo "No GPU: RoMa v2 will run on CPU (slower)"
!git clone -q https://github.com/kavyanshops/project-chitra.git
%cd project-chitra
!pip install -q ".[roma]"

## 1 · Public reference data (LROC NAC DTM + orthophotos, Apollo 11)

In [ ]:
!mkdir -p data/ref
B = "https://pds.lroc.im-ldi.com/data/LRO-L-LROC-5-RDR-V1.0/LROLRC_2001/DATA/SDP/NAC_DTM/APOLLO11"
for f in ["NAC_DTM_APOLLO11.TIF", "NAC_DTM_APOLLO11_M150361817_2M.IMG", "NAC_DTM_APOLLO11_M150368601_2M.IMG"]:
    !wget -q -nc -P data/ref {B}/{f}
!ls -lh data/ref

## 2 · Self-check (known warp → sub-pixel; non-overlap → REJECT)

In [ ]:
!python tests/test_pipeline.py

## 3 · Benchmark with exact ground truth

Each source is the DTM rendered under a *different* Sun and warped by a **known** transform, so **GT RMSE** is exact, in source pixels.
The `REAL-NAC` rows use two real LROC orthophotos with a known warp applied.

In [ ]:
!chitra benchmark --roma > /dev/null
from IPython.display import Markdown
Markdown(open("runs/benchmark.md").read())

## 4 · (Optional) Register your own Chandrayaan-2 crop

On your machine, cut the crop with `scripts/prep_ch2.py` (see `docs/DATA.md`). Then upload `src.tif` and `prep.json` from that output folder.
The reference and DEM windows are cut here from the public LROC files, using the window saved in `prep.json`.
This needs a crop whose reference is the **2 m** orthophoto, as for TMC-2.

In [ ]:
import json, os, subprocess
try:
    from google.colab import files
    up = files.upload()  # select src.tif and prep.json
except ImportError:
    up = {}
if {"src.tif", "prep.json"} <= set(up):
    P = json.load(open("prep.json"))
    c0, r0, w, h = P["ref_window"]
    az, el = P["sun_az_el"]
    cmd = ["chitra", "register", "src.tif", "data/ref/NAC_DTM_APOLLO11_M150361817_2M.IMG",
           "--dem", "data/ref/NAC_DTM_APOLLO11.TIF", "--ref-window", *map(str, (c0, r0, w, h)),
           "--sun-az", str(az), "--sun-el", str(el), "--src-gsd", str(P["gsd_from_grid_m"]), "--name", "colab_upload"]
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)
else:
    print("Skipped: upload src.tif and prep.json to run this step.")

## 5 · Look at the newest run

In [ ]:
import glob, json
from pathlib import Path
from IPython.display import Image, display
from app.main import make_preview
d = Path(sorted(glob.glob("runs/2*"))[-1])
print(d.name, json.dumps(json.loads((d / "metrics.json").read_text()), indent=1))
for k in ("checkerboard", "overlay_matches", "render"):
    if (d / f"{k}.png").exists():
        print(k); display(Image(str(make_preview(d / f"{k}.png")), width=700))